# E1: Thermal-only YOLO baseline

This notebook uses only IR/thermal images and the frozen recording-level split. It does not implement RGB-thermal fusion. The default stage is `validate`; training and held-out test evaluation require an explicit stage change.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import io
import shutil
import json
import os
import random
import tempfile
import zipfile
from collections import Counter
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/UAV_SAR")
if not PROJECT_ROOT.is_dir():
    PROJECT_ROOT = Path.cwd()
SPLIT_MANIFEST = PROJECT_ROOT / "results/dataset_split/recording_split_manifest.csv"
OUTPUT_ROOT = PROJECT_ROOT / "results/thermal_baseline"
PAIR_MANIFEST = OUTPUT_ROOT / "thermal_dataset_manifest.csv"
ZIP_CANDIDATES = [
    Path("/content/drive/MyDrive/WiSARDv1.zip"),
    Path("/content/drive/MyDrive/WiSARD/WiSARDv1.zip"),
]
ZIP_PATH = next((path for path in ZIP_CANDIDATES if path.is_file()), ZIP_CANDIDATES[0])
DRIVE_ROOT = Path("/content/drive/MyDrive/UAV_SAR_E1")

# Safe default: preparation/validation only. Change explicitly to run training.
STAGE = "validate"  # validate | train_cv | train_final | evaluate_test
TOTAL_EPOCHS = 100
CHUNK_EPOCHS = 10
IMAGE_SIZE = 640
BATCH_SIZE = 8
SEED = 20260905
TEST_CONTEXTS = {"210327_Airfield_FLIR", "210812_Hannegan_Enterprise"}
DEVELOPMENT_CONTEXTS = {
    "210417_MtErie_Enterprise",
    "210529_Carnation_Enterprise",
    "210924_FHL_Enterprise",
    "220109_Baker_Enterprise",
}
PAIR_COLUMNS = [
    "image_member_path", "annotation_member_path", "recording_name",
    "recording_path", "collection_context", "partition", "modality",
    "image_mode", "image_width", "image_height", "image_extrema",
]
print(f"Project root: {PROJECT_ROOT}")
print(f"E1 stage: {STAGE}")


In [ ]:
def atomic_write(path: Path, data: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix=f".{path.name}.", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8", newline="") as handle:
            handle.write(data)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)


def manifest_fingerprint(rows: list[dict[str, str]]) -> str:
    payload = "\n".join(
        "|".join(row[column] for column in PAIR_COLUMNS) for row in rows
    )
    return hashlib.sha256(payload.encode()).hexdigest()


def read_frozen_split() -> list[dict[str, str]]:
    if not SPLIT_MANIFEST.is_file():
        raise FileNotFoundError(f"Frozen split is missing: {SPLIT_MANIFEST}")
    with SPLIT_MANIFEST.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    required = {"recording_name", "recording_path", "collection_context", "modality", "partition"}
    if not rows or not required.issubset(rows[0]):
        raise ValueError(f"Frozen split must contain {sorted(required)}")
    thermal = [row for row in rows if row["modality"] == "IR"]
    if not thermal:
        raise ValueError("Frozen split contains no IR recordings.")
    if any(row["partition"] not in {"development", "test"} for row in thermal):
        raise ValueError("Frozen split contains an unsupported partition.")
    return sorted(thermal, key=lambda row: (row["recording_path"], row["recording_name"]))


def validate_annotation(archive: zipfile.ZipFile, member: str) -> tuple[list[int], list[str]]:
    classes, issues = [], []
    try:
        text = archive.read(member).decode("utf-8", errors="replace")
    except (KeyError, OSError, RuntimeError) as error:
        return [], [f"{member}: unreadable annotation: {error}"]
    for line_number, raw in enumerate(text.splitlines(), 1):
        fields = raw.split()
        if not fields:
            continue
        try:
            if len(fields) != 5:
                raise ValueError("expected five YOLO fields")
            class_id = int(fields[0])
            values = [float(value) for value in fields[1:]]
            if class_id < 0 or not all(0 <= value <= 1 for value in values):
                raise ValueError("class or normalized box is out of range")
            classes.append(class_id)
        except ValueError as error:
            issues.append(f"{member}:{line_number}: {error}")
    return classes, issues


def safe_target(root: Path, member: str) -> Path:
    normalized = Path(member.replace("\\", "/"))
    if normalized.is_absolute() or ".." in normalized.parts:
        raise ValueError(f"Unsafe archive member: {member}")
    root_resolved = root.resolve()
    target = (root_resolved / normalized).resolve()
    if target == root_resolved or root_resolved not in target.parents:
        raise ValueError(f"Archive member escapes target: {member}")
    return target


def materialized_image_path(root: Path, row: dict[str, str]) -> Path:
    return safe_target(root / "images" / row["partition"], row["image_member_path"])


def validate_image_list(paths: list[Path]) -> None:
    missing = [str(path) for path in paths if not path.is_file()]
    if missing:
        raise RuntimeError(f"Generated image list contains missing materialized files: {missing[:3]}")


def inspect_image_member(archive: zipfile.ZipFile, member: str) -> dict[str, object]:
    try:
        from PIL import Image
    except ImportError as error:
        raise RuntimeError("Pillow is required for thermal image validation; install the project image stack before preparation.") from error
    payload = archive.read(member)
    try:
        with Image.open(io.BytesIO(payload)) as image:
            image.verify()
        with Image.open(io.BytesIO(payload)) as image:
            extrema = image.getextrema()
            return {"mode": image.mode, "width": image.width, "height": image.height, "extrema": extrema}
    except Exception as error:
        raise ValueError(f"Unreadable thermal image {member}: {error}") from error


def prepare_thermal_manifest() -> list[dict[str, str]]:
    frozen_rows = read_frozen_split()
    if not ZIP_PATH.is_file():
        raise FileNotFoundError(f"Thermal ZIP is missing: {ZIP_PATH}")
    manifest, exclusions, classes = [], [], Counter()
    with zipfile.ZipFile(ZIP_PATH) as archive:
        names = [info.filename.replace("\\", "/") for info in archive.infolist()]
        by_recording: dict[str, list[str]] = {}
        for name in names:
            if Path(name).parts:
                by_recording.setdefault(Path(name).parts[0], []).append(name)
        for frozen in frozen_rows:
            members = by_recording.get(frozen["recording_name"], [])
            image_names = [name for name in members if Path(name).suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}]
            label_names = [name for name in members if Path(name).suffix.lower() in {".txt", ".ann", ".label", ".labels"} and Path(name).name.lower() != "count.txt"]
            images = {}
            labels = {}
            for name in image_names:
                stem = Path(name).stem
                if stem in images:
                    raise ValueError(f"Duplicate thermal image stem: {name}")
                images[stem] = name
            for name in label_names:
                stem = Path(name).stem
                if stem in labels:
                    raise ValueError(f"Duplicate thermal annotation stem: {name}")
                labels[stem] = name
            for stem, image in sorted(images.items()):
                label = labels.get(stem)
                if label is None:
                    exclusions.append({"recording_name": frozen["recording_name"], "image_member_path": image, "reason": "missing annotation"})
                    continue
                try:
                    image_info = inspect_image_member(archive, image)
                except (KeyError, OSError, RuntimeError, ValueError) as error:
                    exclusions.append({"recording_name": frozen["recording_name"], "image_member_path": image, "reason": str(error)})
                    continue
                ids, issues = validate_annotation(archive, label)
                if issues:
                    exclusions.extend({"recording_name": frozen["recording_name"], "image_member_path": image, "reason": issue} for issue in issues)
                    continue
                classes.update(ids)
                manifest.append({
                    "image_member_path": image,
                    "annotation_member_path": label,
                    "recording_name": frozen["recording_name"],
                    "recording_path": frozen["recording_path"],
                    "collection_context": frozen["collection_context"],
                    "partition": frozen["partition"],
                    "modality": "IR",
                    "image_mode": str(image_info["mode"]),
                    "image_width": str(image_info["width"]),
                    "image_height": str(image_info["height"]),
                    "image_extrema": repr(image_info["extrema"]),
                })
    manifest.sort(key=lambda row: (row["partition"], row["recording_path"], row["image_member_path"]))
    if not manifest or not any(row["partition"] == "development" for row in manifest) or not any(row["partition"] == "test" for row in manifest):
        raise ValueError("Thermal preparation did not produce usable development and test samples.")
    csv_buffer = io.StringIO()
    writer = csv.DictWriter(csv_buffer, fieldnames=PAIR_COLUMNS, lineterminator="\n")
    writer.writeheader()
    writer.writerows(manifest)
    atomic_write(PAIR_MANIFEST, csv_buffer.getvalue())
    atomic_write(OUTPUT_ROOT / "thermal_exclusions.json", json.dumps(exclusions, indent=2))
    atomic_write(OUTPUT_ROOT / "thermal_manifest_metadata.json", json.dumps({
        "fingerprint": manifest_fingerprint(manifest),
        "pair_count": len(manifest),
        "class_ids": dict(classes),
        "zip_path": str(ZIP_PATH),
    }, indent=2))
    print(f"Prepared {len(manifest):,} thermal pairs; exclusions: {len(exclusions):,}; classes: {dict(classes)}")
    return manifest


manifest: list[dict[str, str]] = []
if ZIP_PATH.is_file():
    manifest = prepare_thermal_manifest()
    frozen_by_recording = {row["recording_name"]: row for row in read_frozen_split()}
    for row in manifest:
        frozen = frozen_by_recording.get(row["recording_name"])
        if frozen is None or any(row[field] != frozen[field] for field in ("recording_path", "collection_context", "partition")):
            raise ValueError(f"Frozen split mismatch: {row}")
        if row["modality"] != "IR":
            raise ValueError("Non-IR row found in thermal manifest.")
    if {row["collection_context"] for row in manifest} - (TEST_CONTEXTS | DEVELOPMENT_CONTEXTS):
        raise ValueError("Thermal manifest contains an unexpected collection context.")
    print("Thermal manifest validation: PASS")
    print(Counter(row["partition"] for row in manifest))
    print(Counter(row["collection_context"] for row in manifest))
else:
    print(f"Thermal ZIP not found at {ZIP_PATH}; manifest preparation is pending and no data/training action was taken.")


In [ ]:
def materialize(rows: list[dict[str, str]], root: Path) -> None:
    root.mkdir(parents=True, exist_ok=True)
    partition_paths: dict[str, list[Path]] = {"development": [], "test": []}
    with zipfile.ZipFile(ZIP_PATH) as archive:
        for row in rows:
            image_target = materialized_image_path(root, row)
            label_target = safe_target(root / "labels" / row["partition"], row["annotation_member_path"])
            image_target.parent.mkdir(parents=True, exist_ok=True)
            label_target.parent.mkdir(parents=True, exist_ok=True)
            if image_target.exists() and label_target.exists():
                image_bytes = image_target.read_bytes()
                label_bytes = label_target.read_bytes()
                if not image_bytes or not label_bytes:
                    raise RuntimeError(f"Existing materialization is empty: {image_target}")
            else:
                if image_target.exists() or label_target.exists():
                    raise RuntimeError(f"Partial materialization exists for {row['image_member_path']}")
                image_target.write_bytes(archive.read(row["image_member_path"]))
                label_target.write_bytes(archive.read(row["annotation_member_path"]))
            partition_paths[row["partition"]].append(image_target)
    for partition, paths in partition_paths.items():
        validate_image_list(paths)
        atomic_write(root / f"{partition}.txt", "\n".join(str(path) for path in paths) + "\n")


def write_fold_data(rows: list[dict[str, str]], root: Path, validation_context: str) -> None:
    fold_root = root / "fold_data" / validation_context
    fold_root.mkdir(parents=True, exist_ok=True)
    development = [row for row in rows if row["partition"] == "development"]
    if validation_context == "all-development":
        train_rows, validation_rows = development, development
    else:
        train_rows = [row for row in development if row["collection_context"] != validation_context]
        validation_rows = [row for row in development if row["collection_context"] == validation_context]
    if not train_rows or not validation_rows:
        raise ValueError(f"Fold {validation_context} lacks train or validation samples")
    train_paths = [materialized_image_path(root, row) for row in train_rows]
    validation_paths = [materialized_image_path(root, row) for row in validation_rows]
    validate_image_list(train_paths + validation_paths)
    atomic_write(fold_root / "train.txt", "\n".join(str(path) for path in train_paths) + "\n")
    atomic_write(fold_root / "val.txt", "\n".join(str(path) for path in validation_paths) + "\n")
    atomic_write(fold_root / "data.yaml", f"path: {root}\ntrain: {fold_root / 'train.txt'}\nval: {fold_root / 'val.txt'}\nnc: 1\nnames: [person]\n")


def checkpoint_metadata(fold: str) -> dict[str, object]:
    split_fingerprint = hashlib.sha256(SPLIT_MANIFEST.read_bytes()).hexdigest() if SPLIT_MANIFEST.is_file() else "unavailable"
    return {
        "experiment": "E1_thermal_only",
        "fold": fold,
        "manifest_fingerprint": manifest_fingerprint(manifest),
        "split_fingerprint": split_fingerprint,
        "seed": SEED,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "total_epochs": TOTAL_EPOCHS,
    }


def load_checkpoint_payload(path: Path) -> dict | None:
    try:
        import pickle
        import torch
        try:
            payload = torch.load(path, map_location="cpu", weights_only=False)
        except TypeError:
            payload = torch.load(path, map_location="cpu")
        if not isinstance(payload, dict):
            return None
        saved_epoch = payload.get("epoch")
        if not isinstance(saved_epoch, int) or saved_epoch < 0:
            return None
        return payload
    except (EOFError, OSError, RuntimeError, ValueError, pickle.UnpicklingError, AttributeError, ImportError, ModuleNotFoundError):
        return None


def readable_checkpoint(path: Path) -> bool:
    return load_checkpoint_payload(path) is not None


def checkpoint_completed_epochs(path: Path) -> int | None:
    payload = load_checkpoint_payload(path)
    if payload is None:
        return None
    return int(payload["epoch"]) + 1


def find_valid_checkpoint(run_root: Path, fold: str) -> tuple[Path | None, int]:
    metadata_path = run_root / "checkpoint_metadata.json"
    checkpoint_candidates = (run_root / "weights/last.pt", run_root / "weights/last.previous.pt")
    if not metadata_path.is_file():
        if any(candidate.is_file() for candidate in checkpoint_candidates) or any(run_root.iterdir()):
            raise RuntimeError(f"Existing run state lacks compatibility metadata in {run_root}; refusing a fresh restart.")
        return None, 0
    saved = json.loads(metadata_path.read_text(encoding="utf-8"))
    expected = checkpoint_metadata(fold)
    if any(saved.get(key) != value for key, value in expected.items()):
        raise RuntimeError(f"Incompatible checkpoint metadata in {run_root}; refusing a fresh restart.")
    candidates = ((run_root / "weights/last.pt", "completed_epochs"), (run_root / "weights/last.previous.pt", "previous_completed_epochs"))
    metadata_changed = False
    for candidate, epoch_key in candidates:
        if candidate.is_file() and candidate.stat().st_size > 0:
            actual_completed = checkpoint_completed_epochs(candidate)
            if actual_completed is not None:
                if saved.get(epoch_key) != actual_completed:
                    saved[epoch_key] = actual_completed
                    metadata_changed = True
                if metadata_changed:
                    atomic_write(metadata_path, json.dumps(saved, indent=2))
                return candidate, actual_completed
            print(f"Ignoring unreadable or epoch-incomplete checkpoint and trying fallback: {candidate}")
    raise RuntimeError(f"Checkpoint metadata exists but no valid checkpoint is available in {run_root}; refusing pretrained restart.")


def chunk_target(completed: int) -> int:
    return min(completed + CHUNK_EPOCHS, TOTAL_EPOCHS)


def train_selected_fold(fold: str, validation_context: str) -> None:
    if STAGE not in {"train_cv", "train_final"}:
        raise RuntimeError("Training is gated. Set STAGE explicitly to train_cv or train_final.")
    try:
        from ultralytics import YOLO
        import torch
    except ImportError as error:
        raise ImportError("Install the project-pinned Ultralytics environment before training.") from error
    run_root = DRIVE_ROOT / "runs" / fold
    run_root.mkdir(parents=True, exist_ok=True)
    checkpoint, completed = find_valid_checkpoint(run_root, fold)
    if completed >= TOTAL_EPOCHS:
        print(f"{fold} already complete: {completed}/{TOTAL_EPOCHS}")
        return
    random.seed(SEED)
    torch.manual_seed(SEED)
    previous = run_root / "weights/last.previous.pt"
    latest = run_root / "weights/last.pt"
    if checkpoint == latest and latest.is_file():
        shutil.copy2(latest, previous)
        metadata = checkpoint_metadata(fold)
        metadata.update({"completed_epochs": completed, "previous_completed_epochs": completed, "validation_context": validation_context})
        atomic_write(run_root / "checkpoint_metadata.json", json.dumps(metadata, indent=2))
    model = YOLO(str(checkpoint)) if checkpoint else YOLO("yolov8s.pt")
    write_fold_data(manifest, DRIVE_ROOT / "data", validation_context)
    target = chunk_target(completed)
    model.train(data=str(DRIVE_ROOT / "data" / "fold_data" / validation_context / "data.yaml"), project=str(DRIVE_ROOT / "runs"), name=fold, epochs=target, imgsz=IMAGE_SIZE, batch=BATCH_SIZE, seed=SEED, device=0 if torch.cuda.is_available() else "cpu", resume=bool(checkpoint), exist_ok=True)
    latest = run_root / "weights/last.pt"
    actual_completed = checkpoint_completed_epochs(latest)
    if actual_completed is None:
        raise RuntimeError(f"Training returned without a valid epoch-bearing checkpoint: {latest}")
    metadata = checkpoint_metadata(fold)
    metadata.update({"completed_epochs": actual_completed, "previous_completed_epochs": completed, "validation_context": validation_context})
    atomic_write(run_root / "checkpoint_metadata.json", json.dumps(metadata, indent=2))
    print(f"{fold}: completed {actual_completed}/{TOTAL_EPOCHS}; checkpoint={latest}")


if STAGE == "validate":
    print("No training, materialization, or test evaluation requested.")
elif STAGE in {"train_cv", "train_final"}:
    materialize(manifest, DRIVE_ROOT / "data")
    print("Training helpers loaded. Call train_selected_fold explicitly for one fold.")
elif STAGE == "evaluate_test":
    raise RuntimeError("Test evaluation is intentionally gated until development training is complete.")


In [ ]:
from pathlib import Path
import tempfile
import zipfile


def run_e1_regression_smoke_tests() -> None:
    import pickle
    import sys
    import types

    try:
        import torch
        synthetic_torch = torch
    except ImportError:
        synthetic_torch = types.SimpleNamespace(
            save=lambda payload, path: pickle.dump(payload, Path(path).open("wb")),
            load=lambda path, **_: pickle.load(Path(path).open("rb")),
        )
        sys.modules["torch"] = synthetic_torch

    original_zip = globals()["ZIP_PATH"]
    try:
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            archive_path = root / "synthetic.zip"
            rows = [{"image_member_path": "recording/nested/frame_001.png", "annotation_member_path": "recording/nested/frame_001.txt", "recording_name": "recording", "recording_path": "recording", "collection_context": "210417_MtErie_Enterprise", "partition": "development", "modality": "IR", "image_mode": "L", "image_width": "1", "image_height": "1", "image_extrema": "(0, 0)"}]
            with zipfile.ZipFile(archive_path, "w") as archive:
                archive.writestr(rows[0]["image_member_path"], b"thermal")
                archive.writestr(rows[0]["annotation_member_path"], "0 0.5 0.5 0.5 0.5\n")
            globals()["ZIP_PATH"] = archive_path
            materialize(rows, root / "data")
            expected = materialized_image_path(root / "data", rows[0])
            assert expected.is_file()
            assert str(expected) in (root / "data/development.txt").read_text()
            write_fold_data(rows, root / "data", "all-development")
            assert str(expected) in (root / "data/fold_data/all-development/val.txt").read_text()

            run_root = root / "run"
            (run_root / "weights").mkdir(parents=True)
            latest = run_root / "weights/last.pt"
            previous = run_root / "weights/last.previous.pt"
            synthetic_torch.save({"epoch": 14, "model": {"synthetic": True}}, latest)
            synthetic_torch.save({"epoch": 9, "model": {"synthetic": True}}, previous)
            saved = checkpoint_metadata("smoke")
            saved.update({"completed_epochs": 10, "previous_completed_epochs": 10})
            metadata_path = run_root / "checkpoint_metadata.json"
            metadata_path.write_text(json.dumps(saved))

            selected, epoch = find_valid_checkpoint(run_root, "smoke")
            assert selected == latest and epoch == 15
            repaired = json.loads(metadata_path.read_text())
            assert repaired["completed_epochs"] == 15
            assert repaired["previous_completed_epochs"] == 10
            assert chunk_target(epoch) == 25

            latest.write_bytes(b"corrupt")
            selected, epoch = find_valid_checkpoint(run_root, "smoke")
            assert selected == previous and epoch == 10
            assert chunk_target(epoch) == 20

            previous.write_bytes(b"also corrupt")
            try:
                find_valid_checkpoint(run_root, "smoke")
            except RuntimeError:
                pass
            else:
                raise AssertionError("No-valid-checkpoint case restarted silently")
    finally:
        globals()["ZIP_PATH"] = original_zip
        if synthetic_torch.__class__.__module__ == "types":
            sys.modules.pop("torch", None)


# Explicitly call run_e1_regression_smoke_tests() to execute; it is not run automatically.
